In [2]:
!pip install easyocr pdf2image opencv-python-headless
!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [17]:
import os
import cv2
import numpy as np
import json
import easyocr
from pdf2image import convert_from_path
from PIL import Image

# --- CONFIGURATION FOR TEMPORARY STORAGE ---
PDF_INPUT_DIR = "/content/input_pdfs"
JSON_OUTPUT_DIR = "/content/output_results"

# Create the folders in the temporary Colab session
os.makedirs(PDF_INPUT_DIR, exist_ok=True)
os.makedirs(JSON_OUTPUT_DIR, exist_ok=True)

# Initialize EasyOCR (Using GPU for speed)
reader = easyocr.Reader(['en'], gpu=True)

def preprocess_image(image_pil):
    """ Cleans image to make handwriting easier to read """
    img = np.array(image_pil)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Increase contrast and sharpen
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    img = cv2.filter2D(img, -1, kernel)

    # Adaptive thresholding to remove shadows from scans
    img = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY, 11, 2)
    return img

def run_pipeline():
    # Verify files exist
    files = [f for f in os.listdir(PDF_INPUT_DIR) if f.endswith(".pdf")]
    if not files:
        print(f" Error: No PDFs found in {PDF_INPUT_DIR}. Please upload files to the sidebar folder!")
        return

    for pdf_file in files:
        print(f"📄 Extracting: {pdf_file}...")
        pdf_path = os.path.join(PDF_INPUT_DIR, pdf_file)

        # Convert PDF to Images
        try:
            pages = convert_from_path(pdf_path, 300)
        except Exception as e:
            print(f" Could not read PDF: {e}")
            continue

        all_text = []
        for i, page_img in enumerate(pages):
            print(f"  -> Reading Page {i+1}...")
            processed = preprocess_image(page_img)

            # detail=0 gives just text; paragraph=True handles blocks of handwriting
            text_blocks = reader.readtext(processed, detail=0, paragraph=True)
            all_text.append(f"--- Page {i+1} ---\n" + "\n".join(text_blocks))

        # Save result
        result_json = {
            "student_id": pdf_file.replace(".pdf", ""),
            "extracted_content": "\n".join(all_text)
        }

        output_path = os.path.join(JSON_OUTPUT_DIR, pdf_file.replace(".pdf", ".json"))
        with open(output_path, "w") as f:
            json.dump(result_json, f, indent=4)

        print(f"Finished! Text saved to: {output_path}")

# Start the process
run_pipeline()

📄 Extracting: DWM_CIA-1-chp2-DN.pdf...
  -> Reading Page 1...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  -> Reading Page 2...
  -> Reading Page 3...
  -> Reading Page 4...
  -> Reading Page 5...
  -> Reading Page 6...
  -> Reading Page 7...
  -> Reading Page 8...
  -> Reading Page 9...
✅ Finished! Text saved to: /content/output_results/DWM_CIA-1-chp2-DN.json


In [18]:
import json
import pandas as pd
import os

# CONFIGURATION
JSON_INPUT_DIR = "/content/output_results"
PARSED_EXCEL_PATH = "/content/parsed_answers.xlsx"

def extract_answers_from_ocr():
    all_data = []

    # Process every JSON file generated by your OCR
    for file_name in os.listdir(JSON_INPUT_DIR):
        if file_name.endswith(".json"):
            with open(os.path.join(JSON_INPUT_DIR, file_name), 'r') as f:
                data = json.load(f)

            student_id = data.get("student_id", "Unknown")
            raw_text = data.get("extracted_content", "")

            # Logic to split text into Q1, Q2, etc.
            # Splitting by common patterns like "Q1", "Question 2", or "1."
            segments = re.split(r'(Q\d+|Question\s*\d+|\d+\.)', raw_text)

            current_q = None
            for i in range(1, len(segments), 2):
                q_num = re.search(r'\d+', segments[i]).group()
                ans_text = segments[i+1].strip() if i+1 < len(segments) else ""

                all_data.append({
                    "StudentID": student_id,
                    "Question Number": int(q_num),
                    "Answer": ans_text
                })

    df = pd.DataFrame(all_data)
    df.to_excel(PARSED_EXCEL_PATH, index=False)
    print(f"✅ Created Excel for scoring at: {PARSED_EXCEL_PATH}")

# Run this after OCR is done
# extract_answers_from_ocr()

In [ ]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import json

# Load Model (Colab will cache this)
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

def evaluate_semantic_relevance(excel_input, model_answer_json):
    df = pd.read_excel(excel_input)

    # Load your "Teacher" Answer Key
    with open(model_answer_json, 'r') as f:
        answers_key = {str(item["question_number"]): item for item in json.load(f)}

    semantic_scores = []

    for _, row in df.iterrows():
        q_no = str(row["Question Number"])
        student_ans = str(row["Answer"])

        if q_no in answers_key:
            model_ans = answers_key[q_no]["answer"]
            keywords = answers_key[q_no].get("keywords", [])

            # 1. Compute SBERT Similarity
            emb1 = sbert_model.encode(student_ans, convert_to_tensor=True)
            emb2 = sbert_model.encode(model_ans, convert_to_tensor=True)
            sim_score = float(util.cos_sim(emb1, emb2)[0][0])

            # 2. Keyword Check (Basic check)
            found_keywords = sum(1 for k in keywords if k.lower() in student_ans.lower())
            kw_score = found_keywords / len(keywords) if keywords else 1.0

            # Weighted Final Score (80% Meaning, 20% Specific Keywords)
            final_sim = (0.8 * sim_score) + (0.2 * kw_score)
            semantic_scores.append(round(final_sim, 3))
        else:
            semantic_scores.append(0.0)

    df["Final Similarity Score"] = semantic_scores
    # Placeholder columns for Fuzzy Logic
    df["Relevance"] = df["Final Similarity Score"]
    df["Coherence"] = 0.8 # Assume high coherence for now or use your spacy logic

    df.to_excel("/content/final_semantic_scores.xlsx", index=False)
    return "/content/final_semantic_scores.xlsx"

In [ ]:
import skfuzzy as fuzz
import numpy as np

def apply_fuzzy_grading(input_excel):
    df = pd.read_excel(input_excel)

    # Membership functions for 0 to 1 scale
    x_range = np.linspace(0, 1, 11)
    low = fuzz.trimf(x_range, [0, 0, 0.5])
    med = fuzz.trimf(x_range, [0.2, 0.5, 0.8])
    high = fuzz.trimf(x_range, [0.5, 1, 1])

    final_marks = []
    for _, row in df.iterrows():
        score = row["Final Similarity Score"]

        # Simple fuzzy rule: If score is high, marks are high
        # You can expand this with your gaussian_membership logic from your file
        if score > 0.8:
            mark = 0.9 * score # High Performance
        elif score > 0.5:
            mark = 0.7 * score # Average
        else:
            mark = 0.3 * score # Poor

        final_marks.append(round(mark * 10, 2)) # Scaled to 10 marks

    df["Marks"] = final_marks
    df.to_excel("/content/STUDENT_GRADES.xlsx", index=False)
    print("✨ FINAL REPORT GENERATED: /content/STUDENT_GRADES.xlsx")

# apply_fuzzy_grading("/content/final_semantic_scores.xlsx")

In [20]:
!pip install easyocr pdf2image opencv-python-headless sentence-transformers scikit-fuzzy fuzzywuzzy python-Levenshtein
!apt-get install -y poppler-utils


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.7 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [21]:
import os
import cv2
import numpy as np
import json
import re
import pandas as pd
import easyocr
import skfuzzy as fuzz
from pdf2image import convert_from_path
from sentence_transformers import SentenceTransformer, util
from PIL import Image

# --- 1. SETTINGS & PATHS ---
INPUT_PDF_DIR = "/content/input_pdfs"
OUTPUT_JSON_DIR = "/content/ocr_results"
MODEL_ANSWER_PATH = "/content/model_answers.json"
FINAL_REPORT_PATH = "/content/FINAL_EVALUATION_REPORT.xlsx"

os.makedirs(INPUT_PDF_DIR, exist_ok=True)
os.makedirs(OUTPUT_JSON_DIR, exist_ok=True)

# Initialize Models (English OCR + SBERT)
print("🚀 Initializing AI Models...")
reader = easyocr.Reader(['en'], gpu=True)
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

# --- 2. IMAGE PREPROCESSING ---
def clean_image(image_pil):
    img = np.array(image_pil)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # Sharpen and Threshold
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    img = cv2.filter2D(img, -1, kernel)
    img = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    return img

# --- 3. CORE WORKFLOW FUNCTIONS ---

def run_ocr_step():
    print("🔍 Step 1: Running OCR on PDFs...")
    pdf_files = [f for f in os.listdir(INPUT_PDF_DIR) if f.endswith(".pdf")]
    for pdf in pdf_files:
        pages = convert_from_path(os.path.join(INPUT_PDF_DIR, pdf), 300)
        full_text = ""
        for i, page in enumerate(pages):
            processed = clean_image(page)
            text_blocks = reader.readtext(processed, detail=0, paragraph=True)
            full_text += f"\n {' '.join(text_blocks)}"

        with open(os.path.join(OUTPUT_JSON_DIR, pdf.replace(".pdf", ".json")), 'w') as f:
            json.dump({"student_id": pdf.split('.')[0], "text": full_text}, f)

def parse_and_score_step():
    print("🧠 Step 2: Parsing Text & Computing Semantic Similarity...")
    with open(MODEL_ANSWER_PATH, 'r') as f:
        answer_key = {str(item["question_number"]): item for item in json.load(f)}

    all_results = []
    for json_file in os.listdir(OUTPUT_JSON_DIR):
        with open(os.path.join(OUTPUT_JSON_DIR, json_file), 'r') as f:
            data = json.load(f)

        # Split text by common patterns (e.g., Q1, 1., Question 1)
        segments = re.split(r'(Q\d+|Question\s*\d+|\d+\.)', data["text"])

        for i in range(1, len(segments), 2):
            q_num = re.search(r'\d+', segments[i]).group()
            student_ans = segments[i+1].strip() if i+1 < len(segments) else ""

            if q_num in answer_key:
                ref = answer_key[q_num]
                # SBERT Similarity
                sim = float(util.cos_sim(sbert_model.encode(student_ans), sbert_model.encode(ref["answer"]))[0][0])
                # Keyword Bonus
                kw_count = sum(1 for k in ref.get("keywords", []) if k.lower() in student_ans.lower())
                kw_score = kw_count / len(ref["keywords"]) if ref.get("keywords") else 1.0

                final_sim = (0.8 * sim) + (0.2 * kw_score)
                all_results.append({
                    "StudentID": data["student_id"],
                    "Question": q_num,
                    "StudentAnswer": student_ans,
                    "Similarity": round(final_sim, 3)
                })
    return pd.DataFrame(all_results)

def fuzzy_grading_step(df):
    print("⚖️ Step 3: Applying Fuzzy Logic Grading...")
    # Define simple fuzzy marks logic
    def calculate_mark(sim):
        if sim > 0.85: return round(sim * 10, 1)    # High quality
        if sim > 0.50: return round(sim * 8, 1)     # Partial credit
        return round(sim * 4, 1)                    # Poor/Irrelevant

    df["Marks_Obtained"] = df["Similarity"].apply(calculate_mark)
    df["Grade"] = df["Marks_Obtained"].apply(lambda x: 'O' if x>=9 else 'A' if x>=7 else 'B' if x>=5 else 'F')
    df.to_excel(FINAL_REPORT_PATH, index=False)
    print(f"✨ PROCESSED SUCCESSFULLY! Download report at: {FINAL_REPORT_PATH}")

# --- EXECUTION ---
if __name__ == "__main__":
    if not os.path.exists(MODEL_ANSWER_PATH):
        print(f"❌ Error: {MODEL_ANSWER_PATH} not found! Please upload your answer key.")
    else:
        run_ocr_step()
        results_df = parse_and_score_step()
        if not results_df.empty:
            fuzzy_grading_step(results_df)
        else:
            print("❌ No questions were successfully parsed. Check your PDF text format.")

🚀 Initializing AI Models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

❌ Error: /content/model_answers.json not found! Please upload your answer key.
